In [1]:
#Importing relevant libraries..
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import models
from PIL import Image
import matplotlib.pyplot as plt

In [8]:
#Initializing normalizing transform for the dataset
normal_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((32,32)), #resizing to (32*32)
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean = (0.485, 0.456, 0.406),
                                     std = (0.229, 0.224, 0.225))])

#Loading train and test data. Mention the location for extracting images.
train_dir = "/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/vlg-dataset/train"
test_dir = "/kaggle/input/vlg-recruitment-24-challenge/vlg-dataset/vlg-dataset/test"
train_dataset = ImageFolder(root=train_dir, transform=normal_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
print("Class to Index Mapping:", train_dataset.class_to_idx)

Class to Index Mapping: {'antelope': 0, 'bat': 1, 'beaver': 2, 'blue+whale': 3, 'bobcat': 4, 'buffalo': 5, 'chihuahua': 6, 'cow': 7, 'dalmatian': 8, 'deer': 9, 'dolphin': 10, 'elephant': 11, 'german+shepherd': 12, 'giant+panda': 13, 'giraffe': 14, 'grizzly+bear': 15, 'hamster': 16, 'hippopotamus': 17, 'humpback+whale': 18, 'killer+whale': 19, 'leopard': 20, 'lion': 21, 'mole': 22, 'mouse': 23, 'otter': 24, 'ox': 25, 'persian+cat': 26, 'pig': 27, 'polar+bear': 28, 'raccoon': 29, 'rat': 30, 'seal': 31, 'siamese+cat': 32, 'skunk': 33, 'spider+monkey': 34, 'tiger': 35, 'walrus': 36, 'weasel': 37, 'wolf': 38, 'zebra': 39}


In [11]:
#Defining the Model
class CNN32x32(nn.Module):
    def __init__(self):
        super(CNN32x32, self).__init__()
        self.model = nn.Sequential(
            # Layer 1: Input = 3 x 32 x 32, Output = 32 x 32 x 32 (The 3 channels are for RGB)
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),  # 3x3 kernel
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # Output = 32 x 16 x 16 (downsampling)
            
            # Layer 2: Input = 32 x 16 x 16, Output = 64 x 16 x 16
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # Output = 64 x 8 x 8
            
            # Layer 3: Input = 64 x 8 x 8, Output = 128 x 8 x 8
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # Output = 128 x 4 x 4
            
            nn.Flatten(),  # Flatten to feed into fully connected layers
            nn.Linear(128 * 4 * 4, 256),  # Fully connected layer
            nn.ReLU(),
            nn.Linear(256, 40)  # Output layer with 40 units
        )

    def forward(self, x):
        return self.model(x)

#Selecting the appropriate training device 
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNN32x32().to(device)

In [12]:
#Defining the model hyper parameters 
num_epochs = 300
learning_rate = 0.001
weight_decay = 0.01
criterion = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay) 

#Training process begins 
train_loss_list = [] 
for epoch in range(num_epochs): 
	print(f'Epoch {epoch+1}/{num_epochs}:', end = ' ') 
	train_loss = 0
	
	#Iterating over the training dataset in batches 
	model.train() 
	for i, (images, labels) in enumerate(train_loader): 
		
		#Extracting images and target labels for the batch being iterated 
		images = images.to(device) 
		labels = labels.to(device) 

		#Calculating the model output and the cross entropy loss 
		outputs = model(images) 
		loss = criterion(outputs, labels) 

		#Updating weights according to calculated loss 
		optimizer.zero_grad() 
		loss.backward() 
		optimizer.step() 
		train_loss += loss.item() 
	
	#Printing loss for each epoch 
	train_loss_list.append(train_loss/len(train_loader)) 
	print(f"Training loss = {train_loss_list[-1]}") 


Epoch 1/200: Training loss = 3.6401819281753487
Epoch 2/200: Training loss = 3.5099024908199756
Epoch 3/200: Training loss = 3.4487919584165847
Epoch 4/200: Training loss = 3.4109787701763037
Epoch 5/200: Training loss = 3.3885951002305967
Epoch 6/200: Training loss = 3.3733061874989283
Epoch 7/200: 

KeyboardInterrupt: 

In [ ]:
#Prediction..
model.eval()
test_images = [f for f in os.listdir(test_dir) if f.endswith('.jpg')]
test_predictions = []

for img_name in test_images:
    img_path = os.path.join(test_dir, img_name)
    image = Image.open(img_path).convert('RGB')
    image = normal_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        predicted_class = torch.argmax(outputs, dim=1).item()
        test_predictions.append((img_name, train_dataset.classes[predicted_class]))

# Save Predictions
submission = pd.DataFrame(test_predictions, columns=['image_id', 'class'])
submission.to_csv("/kaggle/working/Sub_A1.csv", index=False)#Mention the final saving location